# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** EU General Data Protection Regulation (GDPR), Regulation (EU) 2016/679, official
English text from the Official Journal (88 pages). It is a good stress test because it is full of
general rules followed by exceptions ("shall not apply where..."), prohibitions, and repeated
vocabulary (*consent*, *controller*, *processing*) that shows up in many unrelated passages.

**What this notebook does**

1. Load the PDF and split it into chunks (one shared corpus for both pipelines).
2. Embed chunks with a Bi-Encoder and index them in FAISS.
3. **Pipeline 1 (baseline):** top 10 by cosine similarity → top 3 → LLM → Answer 1.
4. **Pipeline 2 (re-ranking):** same top 10 → Cross-Encoder scores → new top 3 → same LLM prompt → Answer 2.
5. Stress-test both on failure-prone queries and record rank shifts.
6. Profile latency and estimate the cost of running a Cross-Encoder over the full corpus.

Results tables are also written to `results/`.

## 1. Setup

Models:
- **Bi-Encoder:** `sentence-transformers/all-MiniLM-L6-v2` (384-dim, max 256 tokens).
- **Cross-Encoder:** `cross-encoder/ms-marco-MiniLM-L-6-v2` (the model suggested in the brief).
- **LLM:** `gemini-2.5-flash` if a `GEMINI_API_KEY` environment variable is set, otherwise the local
  open-weights model `Qwen/Qwen2.5-3B-Instruct` via Hugging Face. Both use greedy decoding
  (temperature 0) so the only thing that differs between Answer 1 and Answer 2 is the retrieved context.

In [1]:
import json
import os
import re
import statistics
import time
from pathlib import Path

# torch must be imported before faiss: on macOS both ship an OpenMP runtime and importing
# faiss first makes torch segfault on its first forward pass.
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import pymupdf  # a.k.a. fitz

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

PDF_PATH = Path("data/gdpr.pdf")
MODELS_DIR = Path("models")      # optional pre-downloaded copies of the models
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

BI_ENCODER_ID = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LOCAL_LLM_ID = os.environ.get("LOCAL_LLM_ID", "Qwen/Qwen2.5-3B-Instruct")
GEMINI_MODEL_ID = "gemini-2.5-flash"

TOP_K_RETRIEVE = 10   # candidates pulled from the vector store
TOP_K_CONTEXT = 3     # chunks passed to the LLM
MAX_CHUNK_CHARS = 1000  # ~220 tokens, fits inside MiniLM's 256-token window


def model_path(hf_id: str) -> str:
    """Use a local copy under models/ if it exists, otherwise download from the Hugging Face Hub."""
    local = MODELS_DIR / hf_id.split("/")[-1]
    return str(local) if local.exists() else hf_id


LLM_DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("LLM device:", LLM_DEVICE)

LLM device: mps


## 2. Load the PDF

PyMuPDF returns text in layout **blocks**, which in this document line up with paragraphs and list
items. Body text is set in 9.6 pt, footnotes in 8.5 pt and footnote markers in ~5 pt, so font size
is a reliable filter. For every block we:
- drop the running page header (date, "EN", "Official Journal of the European Union", "L 119/xx"),
- drop footnote blocks (mostly < 9 pt text) and the tiny footnote reference markers inside sentences,
- re-join words hyphenated across line breaks,
- keep the page number so answers can be traced back.

In [2]:
HEADER_LINE = re.compile(r"^(\d{1,2}\.\d{1,2}\.\d{4}|EN|Official Journal of the European Union|L \d+/\d+)$")
BODY_MIN_PT = 9.0      # footnotes are 8.5 pt
MARKER_MAX_PT = 7.0    # superscript footnote markers are ~5 pt


def block_text(block) -> str | None:
    """Return cleaned body text for a PyMuPDF 'dict' block, or None if it is a header/footnote."""
    lines, sizes = [], []
    for line in block["lines"]:
        spans = [s for s in line["spans"] if s["size"] > MARKER_MAX_PT]
        text = "".join(s["text"] for s in spans).strip()
        sizes += [s["size"] for s in spans if s["text"].strip()]
        if text and not HEADER_LINE.match(text):
            lines.append(text)
    if not lines or statistics.median(sizes) < BODY_MIN_PT:
        return None
    out = ""
    for line in lines:
        if out.endswith("-") and line[:1].islower():
            out = out[:-1] + line          # "adminis-" + "tration" -> "administration"
        else:
            out = f"{out} {line}" if out else line
    return re.sub(r"\s+", " ", out).strip()


doc = pymupdf.open(PDF_PATH)
blocks = []
for page in doc:
    for block in page.get_text("dict")["blocks"]:
        if block["type"] != 0:  # 0 = text, 1 = image
            continue
        text = block_text(block)
        if text:
            blocks.append({"page": page.number + 1, "text": text})

print(f"{doc.page_count} pages -> {len(blocks)} text blocks")
for b in blocks[300:306]:
    print(b["page"], "|", b["text"][:110])

88 pages -> 1566 text blocks
32 | 3. For the processing of personal data by the Union institutions, bodies, offices and agencies, Regulation (EC
32 | 4. This Regulation shall be without prejudice to the application of Directive 2000/31/EC, in particular of the
32 | Article 3
32 | Territorial scope
32 | 1. This Regulation applies to the processing of personal data in the context of the activities of an establish
33 | 2. This Regulation applies to the processing of personal data of data subjects who are in the Union by a contr


## 3. Chunking

The regulation has two parts: 173 numbered **recitals** (the explanatory preamble) and 99 **articles**
(the binding rules). Chunking decisions:

- **Structure-aware:** a chunk never crosses an article boundary, so each chunk belongs to one rule.
  Every chunk is labelled with its section (e.g. `Article 8 – Conditions applicable to child's consent...`)
  and that label is prepended to the chunk text, the way a reader sees the heading above a paragraph.
- **Size:** paragraphs are packed greedily up to ~1,000 characters (~220 tokens), which fits the
  Bi-Encoder's 256-token limit, so nothing is silently truncated when embedding.
- **Overlap:** when a chunk is full, its last paragraph is carried into the next chunk if it is short
  (< 300 chars). This keeps lead-in sentences ("...unless:") attached to the list that follows.
- Over-long single paragraphs are split at sentence boundaries.
- Recitals are packed together the same way (labelled with the recital numbers they contain).

In [3]:
ARTICLE_RE = re.compile(r"^Article (\d+)$")
STRUCTURAL_RE = re.compile(r"^(CHAPTER [IVX]+|Section \d+)$")
RECITAL_RE = re.compile(r"^\((\d+)\)\s")


def split_sentences(text: str, limit: int) -> list[str]:
    """Split an over-long paragraph into pieces of at most ~limit chars at sentence ends."""
    sentences = re.split(r"(?<=[.;:])\s+(?=[A-Z(])", text)
    pieces, cur = [], ""
    for s in sentences:
        if cur and len(cur) + len(s) + 1 > limit:
            pieces.append(cur)
            cur = s
        else:
            cur = f"{cur} {s}" if cur else s
    if cur:
        pieces.append(cur)
    return pieces


# Pass 1: tag every paragraph with the section it belongs to.
# The articles start at the first block that is exactly "Article 1" (recitals only mention
# articles inline, never as a standalone line).
paragraphs = []
section, in_articles, pending_article, recital = "Preamble", False, None, None
for b in blocks:
    t = b["text"]
    m = ARTICLE_RE.match(t)
    if m and (in_articles or m.group(1) == "1"):
        in_articles = True
        pending_article = m.group(1)        # the next block is the article title
        continue
    if in_articles and pending_article:
        section = f"Article {pending_article} – {t}"
        pending_article = None
        continue
    if STRUCTURAL_RE.match(t) or (in_articles and t.isupper()):
        continue                            # chapter / section headings
    if not in_articles:
        r = RECITAL_RE.match(t)
        if r:
            recital = int(r.group(1))
        section = "Recitals" if recital else "Preamble"
    for piece in split_sentences(t, MAX_CHUNK_CHARS):
        paragraphs.append({"section": section, "page": b["page"], "text": piece, "recital": recital})

# Pass 2: pack paragraphs into chunks within each section.
chunks = []


def emit(section, buf):
    body = " ".join(p["text"] for p in buf)
    label = section
    if section == "Recitals":
        first, last = buf[0]["recital"], buf[-1]["recital"]
        label = f"Recital ({first})" if first == last else f"Recitals ({first})–({last})"
    chunks.append({
        "chunk_id": len(chunks),
        "section": label,
        "page": buf[0]["page"],
        "text": f"{label}\n{body}",
    })


buf = []
for p in paragraphs:
    if buf and (p["section"] != buf[-1]["section"]):
        emit(buf[-1]["section"], buf)
        buf = []
    elif buf and sum(len(x["text"]) for x in buf) + len(p["text"]) > MAX_CHUNK_CHARS:
        emit(buf[-1]["section"], buf)
        buf = [buf[-1]] if len(buf[-1]["text"]) < 300 else []
    buf.append(p)
if buf:
    emit(buf[-1]["section"], buf)

lengths = [len(c["text"]) for c in chunks]
print(f"{len(chunks)} chunks | chars: min {min(lengths)}, median {int(statistics.median(lengths))}, max {max(lengths)}")
chunks_df = pd.DataFrame(chunks)
chunks_df[["chunk_id", "page", "section"]].sample(8, random_state=0).sort_values("chunk_id")

478 chunks | chars: min 127, median 868, max 1433


,chunk_id,page,section
15,15,3,Recitals (17)–(18)
124,124,19,Recitals (103)–(104)
141,141,22,Recital (115)
142,142,22,Recital (116)
155,155,24,Recitals (128)–(129)
188,188,28,Recital (153)
361,361,63,Article 47 – Binding corporate rules
378,378,67,Article 56 – Competence of the lead supervisory authority


## 4. Bi-Encoder embeddings + FAISS index

Each chunk is embedded **independently** of any query. Vectors are L2-normalised, so the inner
product computed by `IndexFlatIP` is exactly cosine similarity. `IndexFlatIP` is exact
(brute-force) search, which is instant for a few hundred chunks and avoids approximation error
muddying the comparison.

In [4]:
bi_encoder = SentenceTransformer(model_path(BI_ENCODER_ID), device="cpu")
texts = [c["text"] for c in chunks]

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(texts, batch_size=32, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
index_build_s = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]}, embedding took {index_build_s:.1f}s")

Indexed 478 chunks, dim=384, embedding took 4.3s


## 5. Retrieval and re-ranking functions

- `retrieve()` – Stage 1: embed the query with the same Bi-Encoder, take the top 10 by cosine similarity.
- `rerank()` – Stage 2: feed each of the 10 `(query, chunk)` pairs **jointly** through the
  Cross-Encoder, which outputs one relevance score per pair; sort descending.

Both return per-stage timings in milliseconds.

In [5]:
cross_encoder = CrossEncoder(model_path(CROSS_ENCODER_ID), device="cpu")


def retrieve(query: str, k: int = TOP_K_RETRIEVE):
    t0 = time.perf_counter()
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    t1 = time.perf_counter()
    scores, ids = index.search(q, k)
    t2 = time.perf_counter()
    candidates = [{"chunk_id": int(i), "bi_rank": r + 1, "bi_score": float(s)}
                  for r, (i, s) in enumerate(zip(ids[0], scores[0]))]
    return candidates, {"query_embed_ms": (t1 - t0) * 1e3, "vector_search_ms": (t2 - t1) * 1e3}


def rerank(query: str, candidates: list[dict]):
    pairs = [(query, chunks[c["chunk_id"]]["text"]) for c in candidates]
    t0 = time.perf_counter()
    ce_scores = cross_encoder.predict(pairs)
    t1 = time.perf_counter()
    reranked = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)
    for r, c in enumerate(reranked):
        c["ce_rank"] = r + 1
    return reranked, {"rerank_ms": (t1 - t0) * 1e3}


# quick sanity check
cands, timing = retrieve("How long does a controller have to report a personal data breach?")
print(timing)
for c in cands[:3]:
    print(c["bi_rank"], round(c["bi_score"], 3), chunks[c["chunk_id"]]["section"])

{'query_embed_ms': 14.932416997908149, 'vector_search_ms': 4.49158300034469}
1 0.674 Article 33 – Notification of a personal data breach to the supervisory authority
2 0.655 Recital (85)
3 0.646 Article 33 – Notification of a personal data breach to the supervisory authority


## 6. LLM and the shared prompt

Both pipelines use **exactly the same prompt template**; only the three context chunks change.
The prompt tells the model to rely only on the context and to say so when the answer is not there,
so a missing chunk shows up as an omission or refusal rather than an answer from the model's own
memory of GDPR.

In [6]:
# The instructions come *after* the context: small models follow instructions placed next to the
# question more reliably than ones buried above ~700 tokens of legal text.
PROMPT_TEMPLATE = """Answer the question about the EU General Data Protection Regulation (GDPR) using only the context passages below.

{context}

Question: {question}

Instructions: Answer in at most 4 sentences, based only on the passages above. Mention any conditions or exceptions that apply and cite the chunk IDs you used, e.g. [Chunk 12]. Only if none of the passages is relevant, reply with exactly "Not found in the provided context." and nothing else.
Answer:"""


def build_prompt(question: str, chunk_ids: list[int]) -> str:
    context = "\n\n".join(
        f"[Chunk {cid} | page {chunks[cid]['page']}]\n{chunks[cid]['text']}" for cid in chunk_ids
    )
    return PROMPT_TEMPLATE.format(context=context, question=question)


if os.environ.get("GEMINI_API_KEY"):
    from google import genai
    from google.genai import types

    _gemini = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    LLM_NAME = GEMINI_MODEL_ID

    def generate(prompt: str) -> str:
        resp = _gemini.models.generate_content(
            model=GEMINI_MODEL_ID, contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=400))
        return resp.text.strip()
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    LLM_NAME = LOCAL_LLM_ID
    _tok = AutoTokenizer.from_pretrained(model_path(LOCAL_LLM_ID))
    _llm = AutoModelForCausalLM.from_pretrained(
        model_path(LOCAL_LLM_ID), torch_dtype=torch.float16 if LLM_DEVICE != "cpu" else torch.float32
    ).to(LLM_DEVICE).eval()

    @torch.no_grad()
    def generate(prompt: str) -> str:
        chat = _tok.apply_chat_template([{"role": "user", "content": prompt}],
                                        tokenize=False, add_generation_prompt=True)
        inputs = _tok(chat, return_tensors="pt").to(LLM_DEVICE)
        out = _llm.generate(**inputs, max_new_tokens=250, do_sample=False,
                            temperature=None, top_p=None, top_k=None,
                            pad_token_id=_tok.eos_token_id)
        return _tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("LLM:", LLM_NAME)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:08<00:08,  8.82s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  7.29s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  7.52s/it]

LLM: Qwen/Qwen2.5-3B-Instruct


## 7. The two pipelines

`run_both()` runs Stage 1 once and feeds the *same* top-10 list to both pipelines, so the comparison
isolates the effect of re-ranking.

In [7]:
def timed_generate(prompt):
    t0 = time.perf_counter()
    ans = generate(prompt)
    return ans, (time.perf_counter() - t0) * 1e3


def pipeline_baseline(query, candidates):
    top = [c["chunk_id"] for c in candidates[:TOP_K_CONTEXT]]
    answer, llm_ms = timed_generate(build_prompt(query, top))
    return top, answer, llm_ms


def pipeline_rerank(query, candidates):
    reranked, t_rr = rerank(query, candidates)
    top = [c["chunk_id"] for c in reranked[:TOP_K_CONTEXT]]
    answer, llm_ms = timed_generate(build_prompt(query, top))
    return reranked, top, answer, llm_ms, t_rr


def run_both(query):
    candidates, t_ret = retrieve(query)
    top1, ans1, llm1 = pipeline_baseline(query, candidates)
    reranked, top2, ans2, llm2, t_rr = pipeline_rerank(query, candidates)
    return {"candidates": reranked, "top_baseline": top1, "top_rerank": top2,
            "answer_1": ans1, "answer_2": ans2,
            "timing": {**t_ret, **t_rr, "llm_1_ms": llm1, "llm_2_ms": llm2}}

## 8. Stress-test queries

The queries were found by reading the regulation, writing ~40 candidate questions aimed at its
traps (exceptions, "shall not", synonyms of legal terms), running both retrievers, and keeping the
ones that show a clear effect. Each query names its **gold passage** by a phrase that only appears
in the chunk(s) that actually answer it, so the evaluation does not depend on hard-coded chunk ids.

Q1–Q6 are cases where the baseline buries the answer and re-ranking promotes it. Q7–Q8 are kept on
purpose as **counter-examples** where re-ranking makes things worse, because the Cross-Encoder is
not uniformly better.

In [8]:
TEST_QUERIES = [
    {"id": "Q1", "type": "Keyword trap / distractor",
     "query": "Does the GDPR apply to someone keeping a personal address book?",
     "gold_phrase": "purely personal or household activity",
     "note": "Everyday words ('address book', 'apply') overlap with the information-duty articles; "
             "the answer uses legal wording ('purely personal or household activity')."},
    {"id": "Q2", "type": "Keyword trap / synonym",
     "query": "Can a decision about me be made only by a computer algorithm?",
     "gold_phrase": "decision based solely on automated processing, including profiling, which produces legal effects",
     "note": "The regulation never says 'algorithm'; Article 22 says 'solely on automated processing'."},
    {"id": "Q3", "type": "Exception / corner case",
     "query": "When does a company not need to tell affected users about a data breach?",
     "gold_phrase": "shall not be required if any of the following conditions are met",
     "note": "The general duty (Art. 33/34(1)) is easy to find; the exemptions in Art. 34(3) sit in a separate chunk."},
    {"id": "Q4", "type": "Exception / corner case",
     "query": "Is a controller exempt from informing people when it got their data from someone else "
              "and contacting them would take disproportionate effort?",
     "gold_phrase": "would involve a disproportionate effort",
     "note": "The exemption is in Art. 14(5), far from the general transparency rules in Art. 12."},
    {"id": "Q5", "type": "Negation & constraints",
     "query": "Which public bodies are not regarded as recipients of personal data?",
     "gold_phrase": "shall not be regarded as recipients",
     "note": "The answer is a 'shall not be regarded' clause hidden inside the Art. 4 definitions list."},
    {"id": "Q6", "type": "Negation & role limits",
     "query": "Can the data protection officer be fired for doing their job?",
     "gold_phrase": "shall not be dismissed or penalised",
     "note": "All DPO articles (37–39) look alike to the Bi-Encoder; only one chunk holds the protection clause."},
    {"id": "Q7", "type": "Counter-example (re-ranking hurts)",
     "query": "Is an employer prohibited from processing health data about employees?",
     "gold_phrase": "in the field of employment and social security and social protection law",
     "note": "Kept to show a failure of the Cross-Encoder."},
    {"id": "Q8", "type": "Counter-example (re-ranking hurts)",
     "query": "Is processing allowed to save someone's life?",
     "gold_phrase": "vital interests of the data subject or of another natural person",
     "note": "Kept to show a failure of the Cross-Encoder."},
]


def gold_chunks(phrase: str) -> set[int]:
    """Chunks whose (whitespace-normalised) text contains the gold phrase."""
    p = phrase.lower()
    return {c["chunk_id"] for c in chunks if p in " ".join(c["text"].split()).lower()}


for q in TEST_QUERIES:
    q["gold"] = gold_chunks(q["gold_phrase"])
    assert q["gold"], f"gold phrase not found for {q['id']}"
pd.DataFrame([{"id": q["id"], "type": q["type"], "query": q["query"],
               "gold chunks": sorted(q["gold"])} for q in TEST_QUERIES])

,id,type,query,gold chunks
0,Q1,Keyword trap / distractor,Does the GDPR apply to someone keeping a personal address book?,"[15, 16, 210]"
1,Q2,Keyword trap / synonym,Can a decision about me be made only by a computer algorithm?,[276]
2,Q3,Exception / corner case,When does a company not need to tell affected users about a data breach?,[305]
3,Q4,Exception / corner case,Is a controller exempt from informing people when it got their data from someone else and contacting them would take...,"[258, 259]"
4,Q5,Negation & constraints,Which public bodies are not regarded as recipients of personal data?,[217]
5,Q6,Negation & role limits,Can the data protection officer be fired for doing their job?,[320]
6,Q7,Counter-example (re-ranking hurts),Is an employer prohibited from processing health data about employees?,[237]
7,Q8,Counter-example (re-ranking hurts),Is processing allowed to save someone's life?,"[228, 237]"


## 9. Run both pipelines on every query

One warm-up call first so model loading and first-call overheads do not land in the measurements.

In [9]:
_ = rerank("warm-up", retrieve("warm-up")[0])
_ = generate("Say OK.")

results = []
for q in TEST_QUERIES:
    r = run_both(q["query"])
    r.update(q)
    results.append(r)
    print(f"{q['id']} done ({(r['timing']['llm_1_ms'] + r['timing']['llm_2_ms']) / 1e3:.1f}s of generation)")

Q1 done (63.3s of generation)


Q2 done (80.5s of generation)


Q3 done (75.0s of generation)


Q4 done (79.6s of generation)


Q5 done (60.2s of generation)


Q6 done (62.0s of generation)


Q7 done (84.7s of generation)


Q8 done (79.8s of generation)


## 10. Rank-shift analysis

For each query: where the best gold chunk sat in the Bi-Encoder top 10 and where the Cross-Encoder
put it. Only ranks 1–3 reach the LLM.

In [10]:
def best_rank(cands, gold, key):
    ranks = [c[key] for c in cands if c["chunk_id"] in gold]
    return min(ranks) if ranks else None


rows = []
for r in results:
    b = best_rank(r["candidates"], r["gold"], "bi_rank")
    c = best_rank(r["candidates"], r["gold"], "ce_rank")
    gold_id = next((x["chunk_id"] for x in sorted(r["candidates"], key=lambda x: x["ce_rank"])
                    if x["chunk_id"] in r["gold"]), None)
    rows.append({
        "id": r["id"], "type": r["type"],
        "gold chunk": f"{gold_id} ({chunks[gold_id]['section'][:45]})" if gold_id is not None else "-",
        "baseline rank": b, "re-ranked rank": c,
        "shift": f"#{b} → #{c}" if b and c else "not in top 10",
        "gold in LLM context (P1)": bool(b and b <= TOP_K_CONTEXT),
        "gold in LLM context (P2)": bool(c and c <= TOP_K_CONTEXT),
    })
rank_shift_df = pd.DataFrame(rows)
rank_shift_df.to_csv(RESULTS_DIR / "rank_shifts.csv", index=False)
rank_shift_df

,id,type,gold chunk,baseline rank,re-ranked rank,shift,gold in LLM context (P1),gold in LLM context (P2)
0,Q1,Keyword trap / distractor,16 (Recital (18)),8,1,#8 → #1,False,True
1,Q2,Keyword trap / synonym,276 (Article 22 – Automated individual decision-ma),4,1,#4 → #1,False,True
2,Q3,Exception / corner case,305 (Article 34 – Communication of a personal data),5,3,#5 → #3,False,True
3,Q4,Exception / corner case,258 (Article 14 – Information to be provided where),4,2,#4 → #2,False,True
4,Q5,Negation & constraints,217 (Article 4 – Definitions),10,2,#10 → #2,False,True
5,Q6,Negation & role limits,320 (Article 38 – Position of the data protection ),6,1,#6 → #1,False,True
6,Q7,Counter-example (re-ranking hurts),237 (Article 9 – Processing of special categories ),4,9,#4 → #9,False,False
7,Q8,Counter-example (re-ranking hurts),228 (Article 6 – Lawfulness of processing),2,5,#2 → #5,True,False


### Full top-10 for each query

`bi_rank`/`bi_score` = Bi-Encoder (cosine), `ce_rank`/`ce_score` = Cross-Encoder (logit, higher is
more relevant). ★ marks the gold chunk(s).

In [11]:
all_cands = []
for r in results:
    for c in sorted(r["candidates"], key=lambda x: x["bi_rank"]):
        all_cands.append({"id": r["id"], "gold": "★" if c["chunk_id"] in r["gold"] else "",
                          "chunk_id": c["chunk_id"], "section": chunks[c["chunk_id"]]["section"][:60],
                          "bi_rank": c["bi_rank"], "bi_score": round(c["bi_score"], 3),
                          "ce_rank": c["ce_rank"], "ce_score": round(c["ce_score"], 2)})
cands_df = pd.DataFrame(all_cands)
cands_df.to_csv(RESULTS_DIR / "top10_candidates.csv", index=False)

for qid, df in cands_df.groupby("id", sort=False):
    print(f"\n{qid}: {next(q['query'] for q in TEST_QUERIES if q['id'] == qid)}")
    print(df.drop(columns="id").to_string(index=False))


Q1: Does the GDPR apply to someone keeping a personal address book?
gold  chunk_id                                                      section  bi_rank  bi_score  ce_rank  ce_score
           254 Article 14 – Information to be provided where personal data         1     0.478        9    -11.23
           250 Article 13 – Information to be provided where personal data         2     0.448        8    -11.20
           257 Article 14 – Information to be provided where personal data         3     0.438        7    -11.17
           258 Article 14 – Information to be provided where personal data         4     0.433        2    -10.99
           270 Article 19 – Notification obligation regarding rectification        5     0.431        3    -11.07
           296                Article 30 – Records of processing activities        6     0.430       10    -11.25
           263             Article 15 – Right of access by the data subject        7     0.429        6    -11.16
   ★        16     

## 11. Answer 1 vs Answer 2

Same prompt, same LLM, same decoding settings — only the three context chunks differ.

In [12]:
def sections(ids):
    return "; ".join(f"{i}: {chunks[i]['section'][:40]}" for i in ids)


answers_md = []
for r in results:
    block = (f"### {r['id']} · {r['type']}\n**Q:** {r['query']}\n\n"
             f"**Pipeline 1 context:** {sections(r['top_baseline'])}\n\n**Answer 1:** {r['answer_1']}\n\n"
             f"**Pipeline 2 context:** {sections(r['top_rerank'])}\n\n**Answer 2:** {r['answer_2']}\n")
    answers_md.append(block)
    print(block.replace("**", "") + "\n" + "-" * 100)

(RESULTS_DIR / "answers.md").write_text(f"# Answers ({LLM_NAME})\n\n" + "\n".join(answers_md))
with open(RESULTS_DIR / "answers.json", "w") as f:
    json.dump([{k: (sorted(v) if isinstance(v, set) else v) for k, v in r.items() if k != "candidates"}
               for r in results], f, indent=2, ensure_ascii=False)

### Q1 · Keyword trap / distractor
Q: Does the GDPR apply to someone keeping a personal address book?

Pipeline 1 context: 254: Article 14 – Information to be provided ; 250: Article 13 – Information to be provided ; 257: Article 14 – Information to be provided 

Answer 1: Not found in the provided context. 

The provided context discusses the GDPR's requirements for providing information to data subjects regarding personal data collection and processing. However, it does not specifically address the situation of keeping a personal address book. Therefore, we cannot determine whether the GDPR applies to someone keeping a personal address book based solely on the given information.

Pipeline 2 context: 16: Recital (18); 258: Article 14 – Information to be provided ; 270: Article 19 – Notification obligation reg

Answer 2: Based on Recital (18) from Chunk 16, the GDPR does not apply to the processing of personal data by a natural person in purely personal or household activities. However

## 12. Latency profiling

Retrieval stages are timed 6 times per query; the first repeat is discarded because it pays a
cold-cache penalty right after the LLM has been using the machine, leaving 8 queries × 5 = 40 runs
per stage. LLM time is taken from the runs above. Pipeline 2 = Pipeline 1's retrieval + re-ranking.
All retrieval models run on CPU; the LLM runs on the Apple-silicon GPU (MPS).

In [13]:
N_REPEATS = 5
lat_rows = []
for q in TEST_QUERIES:
    for rep in range(N_REPEATS + 1):
        cands, t_ret = retrieve(q["query"])
        _, t_rr = rerank(q["query"], cands)
        if rep > 0:
            lat_rows.append({**t_ret, **t_rr})
lat = pd.DataFrame(lat_rows)
llm1 = pd.Series([r["timing"]["llm_1_ms"] for r in results])
llm2 = pd.Series([r["timing"]["llm_2_ms"] for r in results])


def fmt(s):
    digits = 3 if s.mean() < 1 else 1   # FAISS search is sub-millisecond
    return f"{s.mean():.{digits}f} ± {s.std():.{digits}f}"


retrieval = lat["query_embed_ms"] + lat["vector_search_ms"]
latency_df = pd.DataFrame([
    {"stage": "Query embedding (Bi-Encoder)", "Pipeline 1 (ms)": fmt(lat["query_embed_ms"]), "Pipeline 2 (ms)": fmt(lat["query_embed_ms"])},
    {"stage": "Vector search (FAISS, top 10)", "Pipeline 1 (ms)": fmt(lat["vector_search_ms"]), "Pipeline 2 (ms)": fmt(lat["vector_search_ms"])},
    {"stage": "Cross-Encoder re-rank (10 pairs)", "Pipeline 1 (ms)": "–", "Pipeline 2 (ms)": fmt(lat["rerank_ms"])},
    {"stage": "Retrieval total", "Pipeline 1 (ms)": fmt(retrieval), "Pipeline 2 (ms)": fmt(retrieval + lat["rerank_ms"])},
    {"stage": f"LLM generation ({LLM_NAME.split('/')[-1]})", "Pipeline 1 (ms)": fmt(llm1), "Pipeline 2 (ms)": fmt(llm2)},
    {"stage": "End-to-end", "Pipeline 1 (ms)": f"{retrieval.mean() + llm1.mean():.0f}",
     "Pipeline 2 (ms)": f"{retrieval.mean() + lat['rerank_ms'].mean() + llm2.mean():.0f}"},
])
latency_df.to_csv(RESULTS_DIR / "latency.csv", index=False)
print(f"Re-ranking adds {lat['rerank_ms'].mean():.1f} ms, i.e. {lat['rerank_ms'].mean() / retrieval.mean():.1f}x "
      f"the baseline retrieval time, but only {lat['rerank_ms'].mean() / (retrieval.mean() + llm1.mean()) * 100:.1f}% "
      f"of the end-to-end time.")
latency_df

Re-ranking adds 117.6 ms, i.e. 20.0x the baseline retrieval time, but only 0.3% of the end-to-end time.


,stage,Pipeline 1 (ms),Pipeline 2 (ms)
0,Query embedding (Bi-Encoder),5.9 ± 0.4,5.9 ± 0.4
1,"Vector search (FAISS, top 10)",0.042 ± 0.020,0.042 ± 0.020
2,Cross-Encoder re-rank (10 pairs),–,117.6 ± 23.1
3,Retrieval total,5.9 ± 0.4,123.5 ± 23.3
4,LLM generation (Qwen2.5-3B-Instruct),38270.9 ± 9986.0,34858.1 ± 6937.3
5,End-to-end,38277,34982


## 13. Scaling: why not run the Cross-Encoder over the whole corpus?

A Bi-Encoder embeds each chunk **once, offline**; a query then costs one encoder pass plus a vector
search. A Cross-Encoder has nothing to pre-compute, because every score needs the query and the chunk
together, so every query costs one full transformer pass **per chunk**. Below we measure it on this
corpus and extrapolate linearly.

In [14]:
q = TEST_QUERIES[0]["query"]
t0 = time.perf_counter()
full_scores = cross_encoder.predict([(q, c["text"]) for c in chunks], batch_size=32)
full_ce_ms = (time.perf_counter() - t0) * 1e3
per_pair_ms = full_ce_ms / len(chunks)
stage1_ms = retrieval.mean()


def human(seconds):
    for unit, size in [("days", 86400), ("hours", 3600), ("min", 60)]:
        if seconds >= size:
            return f"{seconds / size:,.1f} {unit}"
    return f"{seconds:,.2f} s"


scaling_df = pd.DataFrame([
    {"corpus size (chunks)": f"{n:,}",
     "Cross-Encoder over all chunks, per query": human(per_pair_ms * n / 1e3),
     "Bi-Encoder + top-10 re-rank, per query": human((stage1_ms + lat["rerank_ms"].mean()) / 1e3) + "*"}
    for n in [len(chunks), 10_000, 1_000_000, 100_000_000]
])
scaling_df.to_csv(RESULTS_DIR / "scaling.csv", index=False)
print(f"Measured: {len(chunks)} pairs in {full_ce_ms:.0f} ms = {per_pair_ms:.2f} ms per (query, chunk) pair on CPU")
print("* exact search grows linearly too but is a single matrix product (~microseconds per chunk); "
      "approximate indexes (HNSW/IVF) make it sub-linear")
scaling_df

Measured: 478 pairs in 7489 ms = 15.67 ms per (query, chunk) pair on CPU
* exact search grows linearly too but is a single matrix product (~microseconds per chunk); approximate indexes (HNSW/IVF) make it sub-linear


,corpus size (chunks),"Cross-Encoder over all chunks, per query","Bi-Encoder + top-10 re-rank, per query"
0,478,7.49 s,0.12 s*
1,"10,000",2.6 min,0.12 s*
2,"1,000,000",4.4 hours,0.12 s*
3,"100,000,000",18.1 days,0.12 s*


### Does a full Cross-Encoder pass find anything the two-stage pipeline missed?

In [15]:
full_rows = []
for r in results:
    s = cross_encoder.predict([(r["query"], c["text"]) for c in chunks], batch_size=32)
    full_rank = {int(cid): k + 1 for k, cid in enumerate(np.argsort(-s))}
    full_rows.append({"id": r["id"],
                      "two-stage rank": best_rank(r["candidates"], r["gold"], "ce_rank"),
                      "full Cross-Encoder rank": min(full_rank[g] for g in r["gold"])})
pd.DataFrame(full_rows)

,id,two-stage rank,full Cross-Encoder rank
0,Q1,1,1
1,Q2,1,1
2,Q3,3,4
3,Q4,2,3
4,Q5,2,2
5,Q6,1,1
6,Q7,9,22
7,Q8,5,10
